In [2]:
import re #for regular expressions

def readFile(file_path):
    with open(file_path, 'r', encoding="utf-8") as f:
        input = f.read()

    plaintext = re.sub(r'[^A-Za-z]', '', input) #only keeps letters
    plaintext = plaintext.upper() #convert to uppercase
    return plaintext[:3000]

In [3]:
import random
import string

def generate_random_key(length):
    # all uppercase letters in the alphabet
    letters = string.ascii_uppercase  # "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    key = ""
    for i in range(length):
        # pick one random letter
        random_letter = random.choice(letters)
        key += random_letter # add it to the key

    return key


def vigenere_encryption(plaintext, key):
    # the ASCII code for the upper case letters is: A(65), B(66), ... , Z(90)
    ciphertext = ""
    for i, letter in enumerate(plaintext):
        letter_val = ord(letter) - 65
        key_val = ord(key[i % len(key)]) - 65
        cipher_val = (letter_val + key_val) % 26
        ciphertext += chr(cipher_val + 65)
    return ciphertext


In [4]:
from collections import Counter

def index_of_coincidence(alpha):
    alpha_len = len(alpha)
    if alpha_len == 1:
        return 0
    freq = Counter(alpha) # returneza un dictionar cu frecventa fiecarei litere dintr-un text alpha
    ic = 0
    for f in freq.values():
        ic += f * (f - 1) / (alpha_len * (alpha_len - 1))
    return ic

In [5]:
def determine_key_length(y):
    best_ic = 0
    best_m = 0
    print()
    for m in range(1, 21):
        ic = 0
        for j in range(m):
            ic += index_of_coincidence(y[j::m])
        ic /= m

        if abs(0.065 - ic) < abs(0.065 - best_ic):
            print(f"For m = {m}, IC =  {ic:.4f}")
            if best_ic > 0.06 and ic > 0.06:
                return best_m, best_ic
            best_ic = ic
            best_m = m
    return best_m, best_ic


In [6]:
ENG_FREQ = {
    'A': 0.082, 'B': 0.015, 'C': 0.028, 'D': 0.043, 'E': 0.13,
    'F': 0.022, 'G': 0.02,  'H': 0.061, 'I': 0.07,  'J': 0.0015,
    'K': 0.0077,'L': 0.04,  'M': 0.024, 'N': 0.067, 'O': 0.075,
    'P': 0.019, 'Q': 0.00095,'R': 0.06,  'S': 0.063,'T': 0.091,
    'U': 0.028, 'V': 0.0098,'W': 0.024, 'X': 0.0015,'Y': 0.02, 'Z': 0.00074
}

def mutual_index_of_coincidence(alpha, beta):
    # calculeaza MIC(alpha, beta)
    freq_alpha = Counter(alpha)
    freq_beta = Counter(beta)
    len_alpha, len_beta = len(alpha), len(beta)
    mic = 0
    for letter in string.ascii_uppercase:
        mic += (freq_alpha.get(letter, 0) * freq_beta.get(letter, 0)) / (len_alpha * len_beta)
    return mic

def create_normal_eng_text():
    english_text = ""
    # for each letter in the alphabet (A–Z)
    for letter, freq in ENG_FREQ.items():
        # repeat the letter based on its frequency (for example, 'E' appears 130 times)
        count = int(freq * 1000)
        english_text += letter * count
    return english_text

def shift_text(y, s):
    shifted_text = ""
    for letter in y:
        shifted_text += chr((ord(letter) - 65 - s) % 26 + 65)
    return shifted_text

def determine_key(y, m):
    key = ""
    normal_text = create_normal_eng_text()
    for j in range(m):
        y_col = y[j::m]
        best_diff = float('inf')
        best_s = None
        best_mic = None
        for s in range(26):
            shifted_text = shift_text(y_col, s)
            mic = mutual_index_of_coincidence(normal_text, shifted_text)
            diff = abs(0.065 - mic)
            if diff < best_diff:
                best_diff = diff
                best_s = s
                best_mic = mic
        k_val = best_s 
        key_char = chr(k_val + 65)
        key += key_char
        print(f"Found key character {j}: '{key_char}' (shift = {best_s}, MIC = {best_mic:.5f})")
    return key



In [7]:

def main():
    file_path = "../Substitution Ciphers/emma.txt"
    plaintext = readFile(file_path)

    key = generate_random_key(10)
    print(f"Randomly generated key = {key} | Key length = {len(key)}")
    
    ciphertext = vigenere_encryption(plaintext, key)
    m, ic = determine_key_length(ciphertext)
    print(f"\nDetermined key length = {m}\n")
    print("\nDetermined key = ", determine_key(ciphertext, m))

In [8]:
main()

Randomly generated key = EHJKNJXTYQ | Key length = 10

For m = 1, IC =  0.0409
For m = 2, IC =  0.0438
For m = 4, IC =  0.0441
For m = 5, IC =  0.0530
For m = 10, IC =  0.0680

Determined key length = 10

Found key character 0: 'E' (shift = 4, MIC = 0.06532)
Found key character 1: 'H' (shift = 7, MIC = 0.06723)
Found key character 2: 'J' (shift = 9, MIC = 0.06759)
Found key character 3: 'K' (shift = 10, MIC = 0.06537)
Found key character 4: 'N' (shift = 13, MIC = 0.06775)
Found key character 5: 'J' (shift = 9, MIC = 0.06363)
Found key character 6: 'X' (shift = 23, MIC = 0.06545)
Found key character 7: 'T' (shift = 19, MIC = 0.06991)
Found key character 8: 'Y' (shift = 24, MIC = 0.06617)
Found key character 9: 'Q' (shift = 16, MIC = 0.06841)

Determined key =  EHJKNJXTYQ


In [9]:

def main():
    file_path = "../Substitution Ciphers/emma.txt"
    plaintext = readFile(file_path)

    key = "ABABABABAA"
    print(f"Key = {key} | Key length = {len(key)}")
    
    ciphertext = vigenere_encryption(plaintext, key)
    m, ic = determine_key_length(ciphertext)
    print(f"\nDetermined key length = {m}\n")
    print("\nDetermined key = ", determine_key(ciphertext, m))

main()

Key = ABABABABAA | Key length = 10

For m = 1, IC =  0.0546
For m = 2, IC =  0.0630
For m = 4, IC =  0.0631

Determined key length = 2

Found key character 0: 'A' (shift = 0, MIC = 0.06645)
Found key character 1: 'B' (shift = 1, MIC = 0.06108)

Determined key =  AB


In [10]:

def main():
    file_path = "../Substitution Ciphers/emma.txt"
    plaintext = readFile(file_path)

    key = "ABCABCABCABA"
    print(f"Key = {key} | Key length = {len(key)}")
    
    ciphertext = vigenere_encryption(plaintext, key)
    m, ic = determine_key_length(ciphertext)
    print(f"\nDetermined key length = {m}\n")
    print("\nDetermined key = ", determine_key(ciphertext, m))

main()

Key = ABCABCABCABA | Key length = 12

For m = 1, IC =  0.0477
For m = 2, IC =  0.0480
For m = 3, IC =  0.0629
For m = 6, IC =  0.0642

Determined key length = 3

Found key character 0: 'A' (shift = 0, MIC = 0.06675)
Found key character 1: 'B' (shift = 1, MIC = 0.06747)
Found key character 2: 'C' (shift = 2, MIC = 0.05679)

Determined key =  ABC


In [11]:

def main():
    file_path = "../Substitution Ciphers/emma.txt"
    plaintext = readFile(file_path)

    key = "AAAAAAAB"
    print(f"Key = {key} | Key length = {len(key)}")
    
    ciphertext = vigenere_encryption(plaintext, key)
    m, ic = determine_key_length(ciphertext)
    print(f"\nDetermined key length = {m}\n")
    print("\nDetermined key = ", determine_key(ciphertext, m))

main()

Key = AAAAAAAB | Key length = 8

For m = 1, IC =  0.0612
For m = 2, IC =  0.0621

Determined key length = 1

Found key character 0: 'A' (shift = 0, MIC = 0.06319)

Determined key =  A
